# Temporal split feasibility

Compare January 1 training cutoffs in 2017, 2018, and 2019 for item cold-start recommendation. This analysis uses the full Music4All-Onion v2 population after [preprocessing](01_data_preparation.ipynb).

- Training uses plays before the cutoff. Eligible users played at least 20 distinct tracks with positive tags before that date. The same eligible users apply to validation and test.
- Each cutoff starts a 180-day validation track window. A 30-day gap lets validation outcomes mature before the 180-day test track window begins. All windows use inclusive starts and exclusive ends.
- A candidate track enters a window at its global first observed play. Its target is the set of distinct eligible users who play it in its first 30 days. Future plays provide evaluation labels.
- Every tagged candidate track stays in the comparison, including tracks with zero eligible listeners. The listener-count bands show how much ground truth is available for later evaluation.

The first observed play is a proxy for a track's arrival in the log. Official release and tag availability dates are unknown. The analysis treats the supplied tag snapshot as known when defining user eligibility and scoring tracks.

In [1]:
from datetime import date, timedelta
from pathlib import Path
import sys

import duckdb

root = Path.cwd()
if not (root / "pyproject.toml").is_file():
    root = root.parent
processed = root / "data/processed"
inputs = {name: processed / f"{name}.parquet"
          for name in ("events", "tracks", "track_stats")}
missing = [str(path) for path in inputs.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Run preprocessing first. Missing {', '.join(missing)}")

sys.path.insert(0, str(root))
from scripts.data.inspect_splits import ELIGIBLE_USERS_SQL

db = duckdb.connect(config={
    "threads": 4,
    "memory_limit": "2GB",
    "temp_directory": str(root / "data/tmp"),
})
for name, path in inputs.items():
    db.read_parquet(str(path)).create_view(name)

cutoffs = [date(year, 1, 1) for year in (2017, 2018, 2019)]
window_length = timedelta(days=180)
observation = timedelta(days=30)
windows = []
for cutoff in cutoffs:
    validation_end = cutoff + window_length
    test_start = validation_end + observation
    windows.extend([
        (cutoff, "validation", cutoff, validation_end),
        (cutoff, "test", test_start, test_start + window_length),
    ])

latest_play = db.execute("SELECT max(last_seen) FROM track_stats").fetchone()[0]
if latest_play is None or windows[-1][3] + observation > latest_play.date():
    raise ValueError("The latest test window needs 30 days of follow-up in the log.")

db.execute("""
    CREATE TEMP TABLE candidate_windows (
        cutoff DATE, phase VARCHAR, start_date DATE, end_date DATE
    )
""")
db.executemany("INSERT INTO candidate_windows VALUES (?, ?, ?, ?)", windows);

## Eligible training users

In [2]:
db.execute("CREATE TEMP TABLE eligible_users (cutoff DATE, user_id VARCHAR)")
for cutoff in cutoffs:
    db.execute(
        f"INSERT INTO eligible_users SELECT ?, user_id FROM ({ELIGIBLE_USERS_SQL})",
        [cutoff, cutoff],
    )

db.sql("""
    SELECT w.cutoff, count(u.user_id) AS eligible_users
    FROM candidate_windows AS w
    LEFT JOIN eligible_users AS u USING (cutoff)
    WHERE w.phase = 'validation'
    GROUP BY w.cutoff
    ORDER BY w.cutoff
""").show()

┌────────────┬────────────────┐
│   cutoff   │ eligible_users │
│    date    │     int64      │
├────────────┼────────────────┤
│ 2017-01-01 │         105530 │
│ 2018-01-01 │         105578 │
│ 2019-01-01 │         105611 │
└────────────┴────────────────┘



## Validation and test tracks

The first two counts show all first appearances in each window and those with positive tags. The listener bands and median apply to tagged tracks. Each eligible user contributes once per track.

In [3]:
db.sql("""
    WITH candidates AS (
        SELECT w.cutoff, w.phase, t.track_id, t.first_seen, t.has_tags
        FROM candidate_windows AS w
        JOIN tracks AS t ON t.first_seen >= w.start_date AND t.first_seen < w.end_date
    ), recent_plays AS (
        SELECT user_id, track_id, timestamp
        FROM events
        WHERE timestamp >= ? AND timestamp < ?
    ), listeners AS (
        SELECT c.cutoff, c.phase, c.track_id, c.has_tags,
               count(DISTINCT u.user_id) AS audience
        FROM candidates AS c
        LEFT JOIN recent_plays AS e
          ON c.has_tags AND e.track_id = c.track_id
         AND e.timestamp >= c.first_seen AND e.timestamp < c.first_seen + ?
        LEFT JOIN eligible_users AS u ON u.cutoff = c.cutoff AND u.user_id = e.user_id
        GROUP BY c.cutoff, c.phase, c.track_id, c.has_tags
    )
    SELECT w.cutoff, w.phase, count(l.track_id) AS all_tracks,
           count(*) FILTER (WHERE l.has_tags) AS tagged_tracks,
           count(*) FILTER (WHERE l.has_tags AND audience = 0) AS "0",
           count(*) FILTER (WHERE l.has_tags AND audience BETWEEN 1 AND 9) AS "1-9",
           count(*) FILTER (WHERE l.has_tags AND audience BETWEEN 10 AND 19) AS "10-19",
           count(*) FILTER (WHERE l.has_tags AND audience >= 20) AS "20+",
           median(audience) FILTER (WHERE l.has_tags) AS median_listeners
    FROM candidate_windows AS w
    LEFT JOIN listeners AS l USING (cutoff, phase)
    GROUP BY w.cutoff, w.phase, w.start_date
    ORDER BY w.cutoff, w.start_date
""", params=[cutoffs[0], windows[-1][3] + observation, observation]).show(max_width=120)
db.close()

┌────────────┬────────────┬────────────┬───────────────┬───────┬───────┬───────┬───────┬──────────────────┐
│   cutoff   │   phase    │ all_tracks │ tagged_tracks │   0   │  1-9  │ 10-19 │  20+  │ median_listeners │
│    date    │  varchar   │   int64    │     int64     │ int64 │ int64 │ int64 │ int64 │      double      │
├────────────┼────────────┼────────────┼───────────────┼───────┼───────┼───────┼───────┼──────────────────┤
│ 2017-01-01 │ validation │       2207 │          1204 │     1 │   256 │   133 │   814 │             40.0 │
│ 2017-01-01 │ test       │       2209 │          1137 │     0 │   211 │   159 │   767 │             42.0 │
│ 2018-01-01 │ validation │       2645 │          1274 │     0 │   241 │   128 │   905 │             46.5 │
│ 2018-01-01 │ test       │       3117 │          1299 │     0 │   212 │   166 │   921 │             48.0 │
│ 2019-01-01 │ validation │       1426 │           557 │     0 │    79 │    70 │   408 │             42.0 │
│ 2019-01-01 │ test       │ 

## Recommended split

Use **January 1, 2018** as the training cutoff, subject to confirmation. It gives 105,578 eligible users and the largest tagged cohorts in both windows, with 1,274 validation tracks and 1,299 test tracks. The 2019 cutoff adds 33 eligible users and leaves five tagged tracks in its test window. All three schedules fit within the log's available follow-up.

| Stage | Dates |
| --- | --- |
| Training plays | Before 2018-01-01 |
| Validation track arrivals | [2018-01-01, 2018-06-30) |
| Validation outcomes available and model selection complete by | 2018-07-30 |
| Test track arrivals | [2018-07-30, 2019-01-26) |
| Test follow-up complete by | 2019-02-25 |

Use the fixed pre-cutoff eligible-user pool for both windows. Keep all tagged tracks and report the zero-listener group separately during evaluation. The listener bands can support later analysis of sparse ground truth.